In [1]:
import csv
import hashlib

class User:
    USER_ROLES = {"user", "administrator", "guest"}

    def __init__(self, username, role, password):
        if role not in self.USER_ROLES:
            raise ValueError(f"Invalid role: {role}. Must be one of {self.USER_ROLES}")
        self._username = username
        self._role = role
        self._password_hash = self._hash(password)

    def _hash(self, plain: str) -> str:
        return hashlib.sha256(plain.encode()).hexdigest()

    @property
    def username(self):
        return self._username

    @property
    def role(self):
        return self._role

    def authenticate(self, password: str) -> bool:
        if not isinstance(password, str):
            return False
        return self._password_hash == self._hash(password)

    def can_modify(self, document: 'Document') -> bool:
        if self.role == 'administrator':
            return True
        return document.owner == self

    def to_dict(self):
        return {
            "username": self.username,
            "role": self.role,
            "password": "*hidden*"
        }

    def to_csv_row(self):
        return [self.username, self.role, self._password_hash]

    @staticmethod
    def from_csv_row(row):
        username, role, hashed_pwd = row
        user = User.__new__(User)
        user._username = username
        user._role = role
        user._password_hash = hashed_pwd
        return user

    def __str__(self):
        return f"User({self.username}, {self.role})"



In [2]:
class Document:
    def __init__(self, doc_id, title, content, owner, visibility):
        self.id = int(doc_id)
        self.title = title
        self.content = content
        self.owner = owner
        self.visibility = visibility

    def change_visibility(self, new_visibility, user: User) -> bool:
        if user.can_modify(self):
            self.visibility = new_visibility
            return True
        return False

    def to_dict(self):
        return {
            "id": self.id,
            "title": self.title,
            "content": self.content,
            "owner": self.owner.username,
            "visibility": self.visibility
        }

    def to_csv_row(self):
        return [str(self.id), self.title, self.content, self.owner.username, self.visibility]

    @staticmethod
    def from_csv_row(row, users):
        doc_id, title, content, owner_username, visibility = row
        owner = next((u for u in users if u.username == owner_username), None)
        return Document(doc_id, title, content, owner, visibility)

    def __str__(self):
        return f"Document({self.id}, '{self.title}', {self.visibility})"


class Repository:
    def load(self):
        raise NotImplementedError

    def save(self, items):
        raise NotImplementedError


class UserRepository(Repository):
    def __init__(self, filepath):
        self.filepath = filepath

    def load(self):
        with open(self.filepath, newline='') as csvfile:
            reader = csv.reader(csvfile)
            return [User.from_csv_row(row) for row in reader]

    def save(self, users):
        with open(self.filepath, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            for user in users:
                writer.writerow(user.to_csv_row())


class DocumentRepository(Repository):
    def __init__(self, filepath):
        self.filepath = filepath

    def load(self, users):
        with open(self.filepath, newline='') as csvfile:
            reader = csv.reader(csvfile)
            return [Document.from_csv_row(row, users) for row in reader]

    def save(self, documents):
        with open(self.filepath, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            for doc in documents:
                writer.writerow(doc.to_csv_row())


In [3]:
class Repository:
    def load(self):
        raise NotImplementedError

    def save(self, items):
        raise NotImplementedError


class UserRepository(Repository):
    def __init__(self, filepath):
        self.filepath = filepath

    def load(self):
        with open(self.filepath, newline='') as csvfile:
            reader = csv.reader(csvfile)
            return [User.from_csv_row(row) for row in reader]

    def save(self, users):
        with open(self.filepath, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            for user in users:
                writer.writerow(user.to_csv_row())


class DocumentRepository(Repository):
    def __init__(self, filepath):
        self.filepath = filepath

    def load(self, users):
        with open(self.filepath, newline='') as csvfile:
            reader = csv.reader(csvfile)
            return [Document.from_csv_row(row, users) for row in reader]

    def save(self, documents):
        with open(self.filepath, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            for doc in documents:
                writer.writerow(doc.to_csv_row())


In [4]:

class DocumentManager:
    def __init__(self, user_repo, doc_repo):
        self.user_repo = user_repo
        self.doc_repo = doc_repo
        self.users = user_repo.load()
        self.documents = doc_repo.load(self.users)
        self.current_user = None

    def user_login(self, username, prompted_password):
        for user in self.users:
            if user.username == username:
                if user.authenticate(prompted_password):
                    self.current_user = user
                    return user, True
                else:
                    return user, False
        return None, False

    def get_document_index(self, document_title):
        for index, doc in enumerate(self.documents):
            if doc.title == document_title:
                return index
        return None

    def retrieve_document(self, document_title, connected_user):
        doc_index = self.get_document_index(document_title)
        if doc_index is None:
            return None
        doc = self.documents[doc_index]
        if connected_user.can_modify(doc) or doc.visibility == 'public':
            return doc
        return None

    def change_document_visibility(self, document_title, connected_user, new_visibility):
        doc_index = self.get_document_index(document_title)
        if doc_index is None:
            return False
        doc = self.documents[doc_index]
        return doc.change_visibility(new_visibility, connected_user)


In [8]:

import unittest


class TestUser(unittest.TestCase):
    def setUp(self):
        self.user = User("alice", "user", "secret")

    def test_username(self):
        self.assertEqual(self.user.username, "alice")

    def test_role(self):
        self.assertEqual(self.user.role, "user")

    def test_authenticate_success(self):
        self.assertTrue(self.user.authenticate("secret"))

    def test_authenticate_fail(self):
        self.assertFalse(self.user.authenticate("wrong"))

    def test_invalid_role(self):
        with self.assertRaises(ValueError):
            User("bob", "invalid_role", "pass")


class TestDocument(unittest.TestCase):
    def setUp(self):
        self.owner = User("bob", "user", "pass")
        self.doc = Document(1, "Doc Title", "Some content", self.owner, "private")

    def test_owner_assignment(self):
        self.assertEqual(self.doc.owner.username, "bob")

    def test_change_visibility_by_owner(self):
        self.assertTrue(self.doc.change_visibility("public", self.owner))
        self.assertEqual(self.doc.visibility, "public")

    def test_change_visibility_denied(self):
        outsider = User("eve", "user", "123")
        self.assertFalse(self.doc.change_visibility("private", outsider))


class TestRepositories(unittest.TestCase):
    def setUp(self):
        self.user_repo = UserRepository("test_users.csv")
        self.doc_repo = DocumentRepository("test_docs.csv")

        self.users = [
            User("alice", "user", "alicepwd"),
            User("admin", "administrator", "adminpwd")
        ]
        self.docs = [
            Document(1, "Report", "Details", self.users[0], "private"),
            Document(2, "Overview", "Summary", self.users[1], "public")
        ]
        self.user_repo.save(self.users)
        self.doc_repo.save(self.docs)

    def test_user_load_save(self):
        loaded = self.user_repo.load()
        self.assertEqual(len(loaded), 2)
        self.assertTrue(any(u.username == "alice" for u in loaded))

    def test_document_load_save(self):
        loaded_users = self.user_repo.load()
        loaded_docs = self.doc_repo.load(loaded_users)
        self.assertEqual(len(loaded_docs), 2)
        self.assertEqual(loaded_docs[0].title, "Report")

if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)



..........
----------------------------------------------------------------------
Ran 10 tests in 0.006s

OK


## unittest in Python

Python's built-in unittest framework allows us to systematically verify the correctness of our code. It helps detect bugs early and ensure behavior remains consistent over time (regression testing)

### Why Use unittest?

- Automatically checks your code behaves as expected

- Ensures future changes don't break existing features

- Encourages modular, testable code design

- Acts as live documentation of how the system should behave

### Basic Concepts
A test is a Python class that inherits from `unittest.TestCase`

Each test method should:

- start with the word `test_`

- perform one specific check using assertion methods

Example:
```python
import unittest

class TestMath(unittest.TestCase):
    def test_addition(self):
        self.assertEqual(2 + 2, 4)
```

Run it using:
```bash
python -m unittest filename.py
```

### Test Lifecycle: `setUp()` and `tearDown()`

If your tests need setup, use `setUp()` to prepare shared objects
```python
class TestUser(unittest.TestCase):
    def setUp(self):
        self.user = User("alice", "user", "secret")

    def test_authenticate_success(self):
        self.assertTrue(self.user.authenticate("secret"))
```

### Common Assertions
| Method |	Meaning |
| - | - |
|assertEqual(a, b)	|  Passes if a == b |
|assertTrue(expr)	| Passes if expr is True |
|assertFalse(expr)	| Passes if expr is False |
|assertIsNone(x)	| Passes if x is None |
|assertIsInstance(x, T)	| Passes if x is an instance of type T |
| assertRaises(Error)	| Used to check expected exceptions |


Here’s how we test a User object:
```python
class TestUser(unittest.TestCase):
    def setUp(self):
        self.user = User("bob", "user", "mypwd")

    def test_authenticate_success(self):
        self.assertTrue(self.user.authenticate("mypwd"))

    def test_authenticate_fail(self):
        self.assertFalse(self.user.authenticate("wrong"))
```

### Testing Persistence

To test file operations (CSV load/save), you can write to temporary files and read back:
```python
def test_user_load_save(self):
    self.user_repo.save([User("alice", "user", "123")])
    loaded = self.user_repo.load()
    self.assertEqual(loaded[0].username, "alice")
```

### How to Run the Tests

In your terminal or VSCode:
```bash
python -m unittest test_file.py
```

To run all tests in a directory:
```bash
python -m unittest discover
```


In [9]:
import csv
import hashlib
import os

user_file = "example_users.csv"
doc_file = "example_documents.csv"

if not os.path.exists(user_file):
    with open(user_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["admin", "administrator", hashlib.sha256("adminpwd".encode()).hexdigest()])
        writer.writerow(["alice", "user", hashlib.sha256("alicepwd".encode()).hexdigest()])
        writer.writerow(["bob", "user", hashlib.sha256("bobpwd".encode()).hexdigest()])

if not os.path.exists(doc_file):
    with open(doc_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["1", "Company Strategy", "Top secret...", "admin", "private"])
        writer.writerow(["2", "Public Report", "Available to all.", "admin", "public"])
        writer.writerow(["3", "Alice Notes", "My work log.", "alice", "private"])
        writer.writerow(["4", "Bob Tasks", "Daily todos.", "bob", "private"])

user_repo = UserRepository(user_file)
doc_repo = DocumentRepository(doc_file)
manager = DocumentManager(user_repo, doc_repo)

# Simulate login and viewing documents
print("\n=== AUTHENTICATION AND DOCUMENT RETRIEVAL ===")
user, success = manager.user_login("alice", "alicepwd")
if success:
    print(f"Logged in as {user.username} ({user.role})")
    for doc in manager.documents:
        if user.can_modify(doc) or doc.visibility == 'public':
            print(" -", doc)
else:
    print("Login failed for alice")

print("\nTrying with wrong password:")
_, success = manager.user_login("alice", "wrong")
print("Success:" if success else "Failed")

# Change visibility and demonstrate persistence
print("\nChanging document visibility...")
success = manager.change_document_visibility("Alice Notes", user, "public")
print("Changed successfully:" if success else "Change denied")

doc_repo.save(manager.documents)
print("Documents saved back to file.")

# Reload and verify persistence
print("\n=== RELOADING FROM FILE ===")
new_manager = DocumentManager(user_repo, doc_repo)
for doc in new_manager.documents:
    print(" -", doc)

# Explanation:
# - Users and documents persist in CSV files.
# - Changing visibility affects future retrievals.
# - Simulates long-term state like in a real app DB.



=== AUTHENTICATION AND DOCUMENT RETRIEVAL ===
Logged in as alice (user)
 - Document(2, 'Public Report', public)
 - Document(3, 'Alice Notes', private)

Trying with wrong password:
Failed

Changing document visibility...
Changed successfully:
Documents saved back to file.

=== RELOADING FROM FILE ===
 - Document(1, 'Company Strategy', private)
 - Document(2, 'Public Report', public)
 - Document(3, 'Alice Notes', public)
 - Document(4, 'Bob Tasks', private)
